# 01 - Train and monitor

Trains the model on the configured CSV, with a live dashboard, then evaluates the held-out
TEST block against the baselines. Every number here links to a run directory under `runs/`.
Nothing is defined in this notebook: the logic lives in `neural_trade` (see the README).

In [ ]:
# Parameters
CONFIG_PATH = "../configs/default.yaml"
CSV_PATH = "../binance_btcusdt_1min_ccxt.csv"
RUNS_DIR = "../runs"
OVERRIDES = {}            # e.g. {"EPOCHS": 5, "LR": 5e-4}
EPOCHS = None             # None -> Config.EPOCHS
LIVE_DASHBOARD = True     # ipywidgets dashboard while training
CALIBRATE_LOSS_WEIGHTS = True

In [ ]:
import ipywidgets as widgets
from IPython.display import Markdown, display

from neural_trade.core.config import Config
from neural_trade.data.processor import split_arrays
from neural_trade.evaluation.baselines import BaselineSet
from neural_trade.evaluation.frame import PredictionFrame
from neural_trade.evaluation.report import evaluate
from neural_trade.experiments.run_context import RunContext
from neural_trade.registries.visualizations import Visualizations
from neural_trade.telemetry.epoch_logger import read_metrics
from neural_trade.training.trainer import train_and_evaluate
from neural_trade.visualization.plotly_training import make_interactive_plot_callback

cfg = Config.from_yaml(CONFIG_PATH).override(CSV_PATH=CSV_PATH, **OVERRIDES)
ctx = RunContext.create(cfg, root=RUNS_DIR, tags=["notebook"])
print("run:", ctx.run_dir)

## Train

The dashboard updates every epoch; the per-epoch record is `metrics.jsonl` in the run directory.

In [ ]:
extra_callbacks = []
if LIVE_DASHBOARD:
    loss_out, metrics_out, batch_out = widgets.Output(), widgets.Output(), widgets.Output()
    progress = widgets.IntProgress(min=0, max=EPOCHS or cfg.EPOCHS, description="Epoch")
    display(widgets.VBox([progress, batch_out, loss_out, metrics_out]))
    extra_callbacks.append(make_interactive_plot_callback(
        config=ctx.config, loss_output=loss_out, metrics_output=metrics_out, batch_metrics_output=batch_out,
        progress_widget=progress, total_epochs=EPOCHS or cfg.EPOCHS))

result = train_and_evaluate(config=ctx.config, run_context=ctx, epochs=EPOCHS, force=True,
                            calibrate=CALIBRATE_LOSS_WEIGHTS, fit_calibration=True, save_artifacts=True,
                            extra_callbacks=extra_callbacks)

## Evaluate on the TEST block

Baselines are fitted on the train block; the confidence threshold and calibration come from the cal block.

In [ ]:
blocks = split_arrays(ctx.config)
baselines = BaselineSet.fit(blocks["train"]["X"], blocks["train"]["y"], blocks["train"]["last_close"],
                            ctx.config.DIR_DEADBAND_BPS)
test = PredictionFrame.from_result(result, "test")
cal = PredictionFrame.from_result(result, "cal")
report = evaluate(test, ctx.config, baselines=baselines, cal_frame=cal, run_id=ctx.run_id)
report.to_json(ctx.path("eval_report_test.json"))
display(Markdown(report.to_markdown(ctx.path("eval_report_test.md"))))

In [ ]:
Visualizations.build("eval_report", test, ctx.config).show()

## Training curves and learned indicator periods

In [ ]:
history = read_metrics(ctx.path("metrics.jsonl"))
Visualizations.build("plotly_interactive", history, ctx.config).show()
Visualizations.build("indicator_evolution", ctx.path("metrics.jsonl"), ctx.config).show()